# Biohub Cell Tracking — profiling модели

Полноценное обучение пока не запускается, но узкое место измеряется на **реальном training-step**: чтение данных, forward, losses, backward, optimizer step и NMS.


## Конфигурация

Этот запуск не имеет эпох, DataLoader-цикла, checkpoint или submission. `repeats` определяет только число повторов одного batch для точного измерения.


In [1]:
from __future__ import annotations
import ctypes, json, math, platform, random, sys, time
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import tracksdata as td
import zarr

ROOT=Path.cwd().resolve()
VENDOR=ROOT/"vendor"/"kaggle-cell-tracking-competition"
if not (VENDOR/"src").exists(): raise FileNotFoundError(VENDOR)
sys.path.insert(0,str(VENDOR/"src"))
from tracking_cellmot.models import TemporalUNet3D, SimpleNodeTransformer

@dataclass
class Config:
    # Profiling model training step only; no epoch loop, checkpoints or submission.
    mode: str="profile_training_step"
    stage: str="detector"  # detector | joint; начинаем с детекции, без linker.
    output_dir: str=str(ROOT/"outputs"/"profile")
    dataset_id: str|None=None
    window_start: int=0
    window_size: int=2
    downsample: tuple[int,int,int]=(1,4,4)
    unet_layers: tuple[int,int,int]=(32,64,128)
    gradient_checkpointing: bool=False  # Запас VRAM позволяет убрать повторный forward на backward.
    unet_out_channels: int=32
    transformer_hidden_dim: int=128
    transformer_blocks: int=4
    transformer_pair_chunk_size: int=128  # Профиль 128×128 пар; для плотных кадров выбирать по измерению.
    max_profile_nodes_per_frame: int=128
    learning_rate: float=1e-4
    detection_negative_weight: float=.1
    nms_um: float=5.
    detection_probability: float=.99
    repeats: int=3
    use_amp: bool=True
cfg=Config()
random.seed(42); np.random.seed(42); torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp=cfg.use_amp and device.type=="cuda"
RUN=Path(cfg.output_dir); RUN.mkdir(parents=True,exist_ok=True)
(RUN/"config.json").write_text(json.dumps(asdict(cfg),ensure_ascii=False,indent=2),encoding="utf-8")
print(f"Профилируется training-step модели без эпох/чекпоинтов: {device}, AMP={amp}")


C:\Projects\CellTracker\.venv\Lib\site-packages\numcodecs\__init__.py:106: DeprecationWarning: crc32c usage is deprecated since numcodecs v0.16.4. It is recommended to install google_crc32c instead.
  from numcodecs.checksum32 import CRC32, Adler32, JenkinsLookup3


Профилируется training-step модели без эпох/чекпоинтов: cuda, AMP=True


## Аудит данных

Выбирается одна реальная train-последовательность, но читается лишь окно из двух кадров.


In [2]:
TRAIN=ROOT/"train"
def ids(folder): return [p.stem for p in sorted(folder.glob("*.zarr")) if (folder/f"{p.stem}.geff").exists()]
def load_graph(path):
    value=td.graph.IndexedRXGraph.from_geff(path)
    return value[0] if isinstance(value,tuple) else value
def scale(group): return tuple(dict(group.attrs)["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]["scale"][-3:])
def rss_mib():
    class C(ctypes.Structure): _fields_=[("cb",ctypes.c_ulong),("faults",ctypes.c_ulong),("peak",ctypes.c_size_t),("working",ctypes.c_size_t)]
    c=C(); c.cb=ctypes.sizeof(C); ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.windll.kernel32.GetCurrentProcess(),ctypes.byref(c),c.cb)
    return c.working/2**20
names=ids(TRAIN)
if not names: raise FileNotFoundError(f"Нет train Zarr/GEFF в {TRAIN}")
name=cfg.dataset_id or names[0]
zpath,gpath=TRAIN/f"{name}.zarr",TRAIN/f"{name}.geff"
group=zarr.open_group(zpath,mode="r"); array=group["0"]; graph=load_graph(gpath)
nodes=graph.node_attrs(attr_keys=["node_id","t","z","y","x"]); edges=graph.edge_attrs()
if cfg.window_start<0 or cfg.window_start+1>=array.shape[0]: raise ValueError("Окно вне видео")
audit={"dataset":name,"shape_tzyx":tuple(array.shape),"dtype":str(array.dtype),"chunks":tuple(array.chunks),"scale_zyx_um":scale(group),"nodes":nodes.height,"edges":edges.height,"divisions":int(np.sum(np.asarray(graph.out_degree(graph.node_ids()))>=2))}
pd.DataFrame([audit]).to_csv(RUN/"data_audit.csv",index=False)
audit


{'dataset': '44b6_0113de3b',
 'shape_tzyx': (100, 64, 256, 256),
 'dtype': 'uint16',
 'chunks': (1, 64, 256, 256),
 'scale_zyx_um': (1.625, 0.40625, 0.40625),
 'nodes': 52,
 'edges': 50,
 'divisions': 0}

## Реальный batch

GT-координаты используются только для корректного supervised loss профилировочного training-step, не для inference.


In [3]:
def bench(stage,fn):
    fn(); sync()
    timings=[]; result=None; rss_before=rss_mib()
    if device.type=="cuda": torch.cuda.reset_peak_memory_stats()
    for _ in range(cfg.repeats):
        sync(); started=time.perf_counter(); result=fn(); sync()
        timings.append((time.perf_counter()-started)*1000)
    return result,{"stage":stage,"mean_ms":float(np.mean(timings)),"p50_ms":float(np.median(timings)),"p95_ms":float(np.quantile(timings,.95)),"rss_delta_mib":rss_mib()-rss_before,"peak_vram_mib":torch.cuda.max_memory_allocated()/2**20 if device.type=="cuda" else np.nan}
def sync():
    if device.type=="cuda": torch.cuda.synchronize()
def quantiles():
    q=dict(group.attrs).get("image_statistics",{}).get("quantiles",{})
    if "0.001" in q and "0.999" in q:return float(q["0.001"]),float(q["0.999"])
    return tuple(np.quantile(np.asarray(array[cfg.window_start,::2,::8,::8],dtype="float32"),[.001,.999]))
q0,q1=quantiles(); d=np.asarray(cfg.downsample)
sl=(slice(cfg.window_start,cfg.window_start+2),slice(None,None,d[0]),slice(None,None,d[1]),slice(None,None,d[2]))
raw,r_read=bench("zarr_read_window",lambda:np.asarray(array[sl],dtype="float32"))
images_np,r_norm=bench("normalization",lambda:np.clip((raw-q0)/max(q1-q0,1e-6),0,1))
images=torch.from_numpy(images_np).unsqueeze(0).to(device)
def nodes_at(t):
    frame=nodes.filter(pl.col("t")==t).sort("node_id").head(cfg.max_profile_nodes_per_frame)
    coords=frame.select(["z","y","x"]).to_numpy().astype(np.float32)/d.astype(np.float32)
    return frame,torch.from_numpy(coords).to(device=device, dtype=torch.float32)
a_frame,a=nodes_at(cfg.window_start); b_frame,b=nodes_at(cfg.window_start+1)
if not len(a) or not len(b): raise RuntimeError("В выбранном окне нет двух размеченных кадров")
aid={int(v):i for i,v in enumerate(a_frame["node_id"])}; bid={int(v):i for i,v in enumerate(b_frame["node_id"])}
target=torch.zeros((len(a),len(b)),device=device)
for s,t in zip(edges["source_id"],edges["target_id"],strict=True):
    if int(s) in aid and int(t) in bid: target[aid[int(s)],bid[int(t)]]=1
print("batch",tuple(images.shape),"nodes",len(a),len(b),"supervised edges",int(target.sum()))


batch (1, 2, 64, 64, 64) nodes 1 1 supervised edges 1


## Модель и losses

Используется официальный backbone и transformer; edge loss нормализуется по родителям (`softmax(dim=0)`).


In [4]:
class Tracker(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet=TemporalUNet3D(1,cfg.unet_out_channels,cfg.unet_layers,gradient_checkpointing=cfg.gradient_checkpointing)
        self.detect=nn.Conv3d(cfg.unet_out_channels,1,1)
        self.linker=SimpleNodeTransformer(cfg.unet_out_channels+32,cfg.transformer_hidden_dim,n_blocks=cfg.transformer_blocks,pair_chunk_size=cfg.transformer_pair_chunk_size)
    def index(self,f,c):
        z,y,x=[c[:,i].long().clamp(0,f.shape[i+2]-1) for i in range(3)]
        return f[0,:,z,y,x].T.unsqueeze(0)
    def pos(self,c,t,spatial):
        q=torch.cat([torch.full((len(c),1),t,device=c.device),c],1)
        angle=(q/torch.tensor((2,)+tuple(spatial),device=c.device)).unsqueeze(-1)*(2**torch.arange(4,device=c.device))*math.pi
        return torch.cat([angle.sin(),angle.cos()],-1).flatten(-2).unsqueeze(0)
    def forward(self,x,a,b,with_link=True):
        f=self.unet(x.unsqueeze(2)); da,db=self.detect(f[:,0]),self.detect(f[:,1])
        if not with_link: return da,db,None
        fa,fb=self.index(f[:,0],a),self.index(f[:,1],b)
        ma=torch.ones((1,len(a)),dtype=torch.bool,device=a.device); mb=torch.ones((1,len(b)),dtype=torch.bool,device=b.device)
        logits=self.linker(torch.cat([fa,self.pos(a,0,f.shape[-3:])],-1),torch.cat([fb,self.pos(b,1,f.shape[-3:])],-1),a[None],b[None],ma,mb)[0]
        return da,db,logits
def detector_loss(logits,coords):
    y=torch.zeros_like(logits[:,0]); z,yy,x=[coords[:,i].long().clamp(0,y.shape[i+1]-1) for i in range(3)]; y[0,z,yy,x]=1
    pos=y.sum().clamp_min(1); neg=(y.numel()-pos).clamp_min(1); w=torch.where(y>0,1/pos,torch.full_like(y,cfg.detection_negative_weight)/neg)
    return F.binary_cross_entropy_with_logits(logits[:,0].float(),y.float(),weight=w)
def edge_loss(logits,target):
    rows=target.sum(1)>0; cols=target.sum(0)>0; mask=rows[:,None]|cols[None,:]
    if not mask.any(): return logits.sum()*0
    # BCE вероятностей выполняем в FP32 вне AMP: PyTorch запрещает этот loss в autocast.
    with torch.autocast(device_type=logits.device.type, enabled=False):
        p=torch.softmax(logits.float(),dim=0) # parent normalisation; divisions remain possible
        bce=F.binary_cross_entropy(p,target.float(),reduction="none")
    return (bce*(1-(p*target+(1-p)*(1-target))).pow(2))[mask].mean()
model=Tracker().to(device); optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate); scaler=torch.amp.GradScaler("cuda",enabled=amp)
print("parameters",sum(p.numel() for p in model.parameters()))


parameters 2076706


## Измерение и отчёт

Итоговый `profile.csv` сортирует этапы по времени и включает peak VRAM/RSS. Изменения весов от коротких измерительных шагов не сохраняются.


In [5]:
# Каждый вызов — единичный training-step для измерения. Нет DataLoader epoch, scheduler или checkpoint.
def forward_only():
    with torch.autocast(device_type=device.type,enabled=amp):
        return model(images,a,b,with_link=(cfg.stage == "joint"))
def forward_backward_step():
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type,enabled=amp):
        da,db,logits=model(images,a,b,with_link=(cfg.stage == "joint"))
        loss=detector_loss(da,a)+detector_loss(db,b)
        if cfg.stage == "joint": loss=loss+edge_loss(logits,target)
    if not torch.isfinite(loss): raise FloatingPointError(loss)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(optimizer); scaler.update()
    return loss.detach()
_,r_forward=bench("model_forward",forward_only)
loss,r_step=bench("model_forward_backward_optimizer_step",forward_backward_step)
r_step["loss"]=float(loss)
# NMS cost is measured on actual detector logits; it is outside training-step.
with torch.inference_mode(): det_logits=forward_only()[0]
voxel=np.asarray(scale(group))*d; kernel=tuple(int(v if v%2 else v+1) for v in [max(1,round(cfg.nms_um/s)) for s in voxel])
def nms():
    pool=F.max_pool3d(det_logits,kernel,stride=1,padding=tuple(k//2 for k in kernel))
    return torch.nonzero((det_logits==pool)&(torch.sigmoid(det_logits)>=cfg.detection_probability))
peaks,r_nms=bench("nms",nms)
records=[r_read,r_norm,r_forward,r_step,r_nms]
profile=pd.DataFrame(records).sort_values("mean_ms",ascending=False); profile["share_of_profiled_time"]=profile.mean_ms/profile.mean_ms.sum()
profile.to_csv(RUN/"profile.csv",index=False)
# Sparse GT позволяет честно измерять recall и localisation error, но не полноценный detection precision.
pred=peaks[:,2:].float(); gt=a.float(); spacing=torch.tensor(voxel,device=device,dtype=torch.float32)
if len(pred):
    distance=torch.cdist(pred*spacing,gt*spacing); best_distance,best_gt=distance.min(dim=1); order=best_distance.argsort(); used=set(); matched=[]
    for idx in order.tolist():
        candidate=int(best_gt[idx]); value=float(best_distance[idx])
        if value <= 7.0 and candidate not in used: used.add(candidate); matched.append(value)
else: matched=[]
detection_metrics={"stage":cfg.stage,"sparse_gt_nodes":int(len(gt)),"predicted_peaks":int(len(pred)),"matched_sparse_gt":len(matched),"sparse_gt_recall":len(matched)/len(gt),"mean_localisation_error_um":float(np.mean(matched)) if matched else None,"note":"Sparse GT: recall/localisation are valid diagnostics; peak precision is not a complete-cell precision."}
(RUN/"detection_metrics.json").write_text(json.dumps(detection_metrics,ensure_ascii=False,indent=2),encoding="utf-8")
summary={"dataset":name,"device":str(device),"stage":cfg.stage,"batch_shape":tuple(images.shape),"bottleneck":profile.iloc[0].stage,"bottleneck_mean_ms":float(profile.iloc[0].mean_ms),"peak_count":int(len(peaks)),"detection":detection_metrics}
(RUN/"profile_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
display(profile)
print("Узкое место:",summary["bottleneck"],f"{summary['bottleneck_mean_ms']:.1f} ms")


,stage,mean_ms,p50_ms,p95_ms,rss_delta_mib,peak_vram_mib,loss,share_of_profiled_time
3,model_forward_backward_optimizer_step,96.834567,96.8720,96.88163,0.0,3047.105957,9.837079e-08,0.685243
2,model_forward,24.469867,23.3956,26.87635,0.0,2610.861328,NaN,0.173159
0,zarr_read_window,18.783900,18.6933,19.06329,0.0,0.000000,NaN,0.132923
1,normalization,1.034000,1.0432,1.04770,0.0,0.000000,NaN,0.007317
4,nms,0.191967,0.1730,0.24455,0.0,665.735352,NaN,0.001358


Узкое место: model_forward_backward_optimizer_step 96.8 ms


## Следующий выбор

После измерений оптимизируйте только доминирующий этап: chunks/кэш для Zarr, spatial profile для U-Net, либо генерацию кандидатов по физическому радиусу для linker. Затем повторите точно тот же профиль.
